# CEG Colab 端到端论文结果包流水线

该 Notebook 在一个独立 Colab 冷启动会话中完成 prompt → 图像生成 → attack → detection → fixed-FPR → 论文结果包归档。它不读取其他 Colab 会话的 `/content` 中间结果; 若需要跨会话接续, 请使用分阶段 Notebook。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG.git"
REPO_BRANCH = ""
REPO_DIR = Path("/content/CEG")
DRIVE_ROOT = Path("/content/drive/MyDrive/CEG")
LOCAL_RUNTIME_ROOT = Path("/content/ceg_runtime")
PROFILE = "paper_main_probe"
RUN_ID = f"{PROFILE}_end_to_end_paper_pipeline"
WORKSPACE = LOCAL_RUNTIME_ROOT / RUN_ID
RESET_LOCAL_RUNTIME_WORKSPACE = True

# 正式运行配置: True 表示本 Notebook 会调用真实 SD 与 CEG watermark backend。
RUN_IMAGE_GENERATION = True
SD_MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
HF_TOKEN_ENV = "HF_TOKEN"
HF_CACHE_ROOT = LOCAL_RUNTIME_ROOT / "huggingface_cache"
WATERMARK_BACKEND = "ceg_content_chain_embedding"
SEMANTIC_MASK_BACKEND = "ceg_inspyrenet_semantic_mask"
ATTESTATION_KEY_ENV = "CEG_ATTESTATION_KEY"
ATTESTATION_KEY_ID = "formal_colab_run_key"
DETECTION_FORMAL_RESULT_CLAIM = True
PREPARE_INSPYRENET_WEIGHT = True
INSPYRENET_WEIGHT_URL = "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth"

# 论文结果包配置。
ATTACK_FAMILIES = "brightness_contrast,gaussian_noise,rotate,resize,jpeg"
TARGET_FPR = "0.01"
ALLOW_INCOMPLETE_PACKAGE = False
ALLOW_INVALID_ARCHIVE = False
REFRESH_STAGE_SUMMARY = True

PROMPT_PLAN = REPO_DIR / "prompts" / "prompt_plans" / f"{PROFILE}_prompt_plan.json"
MODEL_CONFIG = WORKSPACE / "configs" / "model_config.draft.json"
INSPYRENET_WEIGHT_DRIVE_PATH = Path("/content/drive/MyDrive/Models/inspyrenet/ckpt_base.pth")
IMAGE_OUTPUT_ROOT = WORKSPACE / "inputs" / "images"
PIPELINE_OUT = WORKSPACE / "paper_end_to_end_pipeline"


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"非 Colab 环境或 Drive 已挂载: {exc}")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    cmd = ["git", "clone"]
    if REPO_BRANCH:
        cmd += ["--branch", REPO_BRANCH]
    cmd += [REPO_URL, str(REPO_DIR)]
    print("运行:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"], check=True)
    if REPO_BRANCH:
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate", "safetensors", "Pillow", "PyYAML", "huggingface_hub", "transparent-background"], check=True)

from paper_workflow.colab_utils.runtime import (
    ensure_attestation_key,
    prepare_huggingface_snapshot,
    prepare_inspyrenet_weight,
    write_model_config_with_cache,
)

if WORKSPACE.exists() and RESET_LOCAL_RUNTIME_WORKSPACE:
    shutil.rmtree(WORKSPACE)
(WORKSPACE / "configs").mkdir(parents=True, exist_ok=True)
IMAGE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ensure_attestation_key(ATTESTATION_KEY_ENV)
if PREPARE_INSPYRENET_WEIGHT:
    prepare_inspyrenet_weight(
        drive_weight_path=INSPYRENET_WEIGHT_DRIVE_PATH,
        weight_url=INSPYRENET_WEIGHT_URL,
    )

snapshot_summary = prepare_huggingface_snapshot(
    model_id=SD_MODEL_ID,
    hf_token_env=HF_TOKEN_ENV,
    cache_root=HF_CACHE_ROOT,
)
write_model_config_with_cache(
    path=MODEL_CONFIG,
    model_id=SD_MODEL_ID,
    snapshot_summary=snapshot_summary,
)

required_inputs = [WORKSPACE, PROMPT_PLAN, MODEL_CONFIG]
missing = [str(path) for path in required_inputs if not path.exists()]
if missing:
    raise FileNotFoundError("端到端流水线缺少输入: " + ", ".join(missing))
print("workspace =", WORKSPACE)
print("prompt_plan =", PROMPT_PLAN)
print("model_config =", MODEL_CONFIG)


In [ ]:
print("端到端流水线已经完成冷启动、依赖安装、模型 snapshot 预检和工作区创建。")


In [ ]:
print("本 Notebook 不读取 Google Drive 前序工作区。跨会话运行请使用分阶段 Notebook 和 Drive 阶段归档 zip。")


In [ ]:
cmd = [
    sys.executable,
    "scripts/run_colab_end_to_end_paper_pipeline.py",
    "--workspace", str(WORKSPACE),
    "--drive-root", str(DRIVE_ROOT),
    "--run-id", RUN_ID,
    "--out", str(PIPELINE_OUT),
    "--prompt-plan", str(PROMPT_PLAN),
    "--model-config", str(MODEL_CONFIG),
    "--image-output-root", str(IMAGE_OUTPUT_ROOT),
    "--sd-model-id", SD_MODEL_ID,
    "--hf-token-env", HF_TOKEN_ENV,
    "--watermark-backend", WATERMARK_BACKEND,
    "--semantic-mask-backend", SEMANTIC_MASK_BACKEND,
    "--attestation-key-env", ATTESTATION_KEY_ENV,
    "--attestation-key-id", ATTESTATION_KEY_ID,
    "--attack-families", ATTACK_FAMILIES,
    "--target-fpr", TARGET_FPR,
    "--profile", PROFILE,
]
if RUN_IMAGE_GENERATION:
    cmd.append("--run-image-generation")
if DETECTION_FORMAL_RESULT_CLAIM:
    cmd.append("--detection-formal-result-claim")
if ALLOW_INCOMPLETE_PACKAGE:
    cmd.append("--allow-incomplete-package")
if ALLOW_INVALID_ARCHIVE:
    cmd.append("--allow-invalid-archive")
if REFRESH_STAGE_SUMMARY:
    cmd.append("--refresh-stage-summary")
print("运行:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
manifest_path = PIPELINE_OUT / "colab_end_to_end_paper_pipeline_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({
    "overall_decision": manifest.get("overall_decision"),
    "image_generation_archive_zip": manifest.get("image_generation_archive_zip"),
    "paper_results_package_root": manifest.get("paper_results_package_root"),
    "paper_pipeline_manifest": manifest.get("paper_pipeline_manifest"),
}, ensure_ascii=False, indent=2))
if manifest.get("overall_decision") != "pass":
    raise RuntimeError("端到端论文结果包流水线未通过。")

# 对本次端到端运行做正式验收。正式 GPU 运行不应开启 allow-existing-image-generation。
formal_acceptance_report = PIPELINE_OUT / "colab_end_to_end_formal_run_acceptance_report.json"
acceptance_cmd = [
    sys.executable,
    "scripts/validate_colab_end_to_end_formal_run.py",
    "--manifest", str(manifest_path),
    "--out", str(formal_acceptance_report),
    "--require-evidence",
    "--require-image-examples",
    "--require-pass",
]
subprocess.run(acceptance_cmd, check=True)
formal_acceptance = json.loads(formal_acceptance_report.read_text(encoding="utf-8"))
print(json.dumps({
    "formal_acceptance_decision": formal_acceptance.get("overall_decision"),
    "formal_acceptance_report": str(formal_acceptance_report),
}, ensure_ascii=False, indent=2))
